# Statistical Arbitrage Pairs Trading with Cointegration and Risk Controls
**The Thalesians | Quantitative Research | Deliverable 1**  
**Universe:** JPM, BAC, C, GS, MS | **Frequency:** Daily adjusted prices | **Study:** 2018–2025

> **Research question:** Does an economically related equity pair exhibit a tradable, out-of-sample mean-reversion signal **after** transaction costs and risk controls?

**Research integrity:** This notebook downloads real data when executed. No performance statistics are pre-populated or fabricated. Results depend on the data vendor, run date and parameters. The initial configuration is a research starting point, not an optimized trading strategy.

## 0. Reproducibility and methodology
- **Training (2018–2021):** compare all 10 pairs using Engle–Granger cointegration; select the lowest p-value *within training only*.
- **Validation (2022–2023):** choose entry threshold / lookback using net Sharpe. No test-period selection.
- **Test (2024–2025):** one untouched, chronological out-of-sample evaluation.
- **Signal formation:** rolling hedge ratio, spread mean and standard deviation are estimated with data available **through t**; trade at the next close-to-close interval (t+1). This simplified daily-close convention does not model next-open execution or overnight gaps.
- **Portfolio returns:** shares of Y and X are proportional to +1 and −β, normalized by gross marked-to-market notional. Positions are recomputed each day; this is a daily rebalanced synthetic portfolio, **not** a static-share pair. Costs reflect daily weight changes and position entry/exit.
- **Caveats:** adjusted closes are total-return-adjusted proxies and cannot substitute for executable bid/ask quotes; short borrow fees, financing, slippage and market impact are not included. Rolling parameter updates, close execution and corporate actions require additional production controls.


In [ ]:
# Uncomment if needed in a fresh environment:
# %pip install yfinance pandas numpy scipy statsmodels matplotlib seaborn
import itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from scipy.stats import norm
warnings.filterwarnings('ignore', category=FutureWarning)
OUT = Path('pairs_trading_outputs'); OUT.mkdir(exist_ok=True)
TICKERS = ['JPM','BAC','C','GS','MS']
START, END = '2018-01-01', '2026-01-01'  # end exclusive
TRAIN_END, VAL_END = '2021-12-31', '2023-12-31'
TRADING_DAYS = 252
print('Output folder:', OUT.resolve())

## 1. Data preparation
Download auto-adjusted daily close prices. Keep a common trading-date intersection, report missing values and save the aligned dataset. **An internet connection is required for the first run.**

In [ ]:
import yfinance as yf
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True,
                  progress=False, group_by='column', threads=False)
if raw.empty: raise RuntimeError('No data returned. Check internet connectivity / Yahoo Finance access.')
prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']].rename(columns={'Close':TICKERS[0]})
prices = prices.reindex(columns=TICKERS).sort_index()
missing = prices.isna().sum().rename('missing_days')
print('Missing observations before alignment:'); display(missing.to_frame())
prices = prices.dropna(how='any')
if len(prices) < 1000: raise ValueError('Too few aligned daily prices for the proposed study.')
assert prices.index.is_monotonic_increasing and prices.notna().all().all()
prices.to_csv(OUT/'bank_prices_aligned.csv', index_label='Date')
print('Aligned sample:',prices.index.min().date(),'to',prices.index.max().date(), '| rows:',len(prices))
display(prices.head())

## 2. Candidate-pair selection — training data only
The Engle–Granger test is applied to price levels with an intercept. Correlation is descriptive and **not** a substitute for cointegration. A low p-value is exploratory evidence, not proof of future tradability. Screening 10 pairs creates multiple-testing risk.

In [ ]:
train = prices.loc[:TRAIN_END]
validation = prices.loc['2022-01-01':VAL_END]
test = prices.loc['2024-01-01':]
assert len(train)>200 and len(validation)>200 and len(test)>200
rows=[]
for x,y in itertools.combinations(TICKERS,2):
    # Convention: Y = y, X = x; direction fixed by alphabetical universe ordering.
    yy,xx = train[y],train[x]
    beta, alpha = np.polyfit(xx.to_numpy(), yy.to_numpy(), 1)
    stat,pvalue,_ = coint(yy,xx,trend='c',autolag='aic')
    resid=yy-alpha-beta*xx
    rows.append(dict(X=x,Y=y,correlation=xx.corr(yy),alpha=alpha,beta=beta,
                     coint_stat=stat,coint_pvalue=pvalue,
                     spread_adf_stat=adfuller(resid,autolag='AIC')[0]))
selection=pd.DataFrame(rows).sort_values('coint_pvalue').reset_index(drop=True)
selection.insert(0,'rank',np.arange(1,len(selection)+1))
selection.to_csv(OUT/'pair_selection_training.csv',index=False)
display(selection.round(5))
X,Y=selection.loc[0,['X','Y']]
print(f'Selected training pair: Y={Y}, X={X}; cointegration p={selection.loc[0,"coint_pvalue"]:.4g}')
if selection.loc[0,'coint_pvalue']>=0.05:
    print('WARNING: No candidate passes the exploratory 5% cointegration threshold. Treat any subsequent strategy as a negative-control / exploratory result, NOT validated cointegration.')

## 3. Walk-forward hedge ratio and signals
At each date, regress Y on X using the **trailing** window, then compute the current spread. The rolling spread mean/std are based on the historical spreads generated by the same causal procedure. Signals are acted on **one trading day later**. A fixed historical hedge ratio is also a legitimate alternative; here we deliberately use rolling estimation as requested.

In [ ]:
def rolling_beta(y,x,lookback):
    # rolling covariance/variance, using current and past observations only
    b=y.rolling(lookback,min_periods=lookback).cov(x) / x.rolling(lookback,min_periods=lookback).var()
    a=y.rolling(lookback,min_periods=lookback).mean()-b*x.rolling(lookback,min_periods=lookback).mean()
    return a,b

def build_signals(px, y_name, x_name, lookback=120, entry=2.0, exit=0.5,
                  stop_z=4.0, max_hold=20):
    y,x=px[y_name],px[x_name]
    alpha,beta=rolling_beta(y,x,lookback)
    spread=y-alpha-beta*x
    mu=spread.rolling(lookback,min_periods=lookback).mean()
    sd=spread.rolling(lookback,min_periods=lookback).std()
    z=((spread-mu)/sd.replace(0,np.nan)).replace([np.inf,-np.inf],np.nan)
    signal=pd.Series(0.0,index=px.index)
    hold=0; position=0
    # Positions in signal[t] are targets computed at t; used from t to t+1.
    for t in range(len(px)):
        zt=z.iloc[t]
        if not np.isfinite(zt) or not np.isfinite(beta.iloc[t]):
            position=0;hold=0
        elif position:
            hold+=1
            if abs(zt)<=exit or abs(zt)>=stop_z or hold>=max_hold:
                position=0;hold=0
        elif zt>=entry:
            position=-1;hold=0
        elif zt<=-entry:
            position=1;hold=0
        signal.iloc[t]=position
    return pd.DataFrame({'Y':y,'X':x,'alpha':alpha,'beta':beta,
                         'spread':spread,'zscore':z,'signal':signal})

## 4. Backtesting, cost model and risk metrics
The daily pair weight at signal time is \(w_Y=s\,P_Y/(P_Y+|\beta|P_X)\), \(w_X=-s\beta P_X/(P_Y+|\beta|P_X)\). The next daily return is \(w_Y r_Y+w_X r_X\). A transaction-cost charge of **bps × absolute turnover of both weights** is deducted. Weight normalization gives approximately unit gross exposure at each rebalance. The strategy is assumed to earn zero on unused cash; funding and borrow costs are excluded.

In [ ]:
def backtest(px, y_name, x_name, lookback=120, entry=2., exit=0.5,
             stop_z=4., max_hold=20, cost_bp=5.):
    f=build_signals(px,y_name,x_name,lookback,entry,exit,stop_z,max_hold)
    denom=f['Y']+f['beta'].abs()*f['X']
    wy=(f['signal']*f['Y']/denom).fillna(0)
    wx=(-f['signal']*f['beta']*f['X']/denom).fillna(0)
    # target weights formed on date t; next row's return represents t -> t+1
    ry=f['Y'].pct_change().fillna(0)
    rx=f['X'].pct_change().fillna(0)
    gross=(wy.shift(1).fillna(0)*ry+wx.shift(1).fillna(0)*rx)
    turnover=(wy.diff().abs().fillna(wy.abs())+wx.diff().abs().fillna(wx.abs()))
    # charge for the weights decided at t on the next return observation
    cost=(turnover.shift(1).fillna(0)*cost_bp/10000)
    f['wY'],f['wX']=wy,wx
    f['turnover']=turnover.shift(1).fillna(0)
    f['gross_return']=gross
    f['transaction_cost']=cost
    f['net_return']=gross-cost
    return f

def metrics(df, ret_col='net_return'):
    r=df[ret_col].dropna()
    if not len(r): return {}
    nav=(1+r).cumprod(); dd=nav/nav.cummax()-1
    ann=(nav.iloc[-1]**(TRADING_DAYS/len(r))-1) if nav.iloc[-1]>0 else np.nan
    vol=r.std()*np.sqrt(TRADING_DAYS)
    downside=np.sqrt((np.minimum(r,0)**2).mean())*np.sqrt(TRADING_DAYS)
    var5=r.quantile(.05); cvar5=r[r<=var5].mean()
    return {'annual_return':ann,'annual_vol':vol,
            'sharpe':r.mean()/r.std()*np.sqrt(TRADING_DAYS) if r.std()>0 else np.nan,
            'sortino':r.mean()*TRADING_DAYS/downside if downside>0 else np.nan,
            'max_drawdown':dd.min(),'worst_day':r.min(),'VaR_95_daily':-var5,
            'CVaR_95_daily':-cvar5,'total_cost':df['transaction_cost'].sum(),
            'avg_daily_turnover':df['turnover'].mean(),
            'trade_entries':int(((df['signal']!=0)&(df['signal'].shift(1).fillna(0)==0)).sum())}

# Small unit checks: no signal before warmup and no NaN returns.
_demo=backtest(prices.iloc[:500],Y,X)
assert _demo['net_return'].notna().all()
assert (_demo['signal'].iloc[:119]==0).all()
print('Basic backtest checks passed.')

## 5. Validation-only parameter selection
For fair warm-up, calculate indicators using earlier data but **score only 2022–2023**. Parameters are chosen using validation-period net Sharpe. The test period is not consulted.

In [ ]:
parameter_rows=[]
for lookback,entry in itertools.product([60,120,252],[1.5,2.,2.5]):
    run=backtest(prices.loc[:VAL_END],Y,X,lookback=lookback,entry=entry,cost_bp=5)
    val_run=run.loc['2022-01-01':VAL_END]
    m=metrics(val_run)
    parameter_rows.append({'lookback':lookback,'entry':entry,**m})
validation_table=pd.DataFrame(parameter_rows).sort_values('sharpe',ascending=False,na_position='last')
validation_table.to_csv(OUT/'validation_parameter_search.csv',index=False)
display(validation_table[['lookback','entry','annual_return','sharpe','max_drawdown','trade_entries']].round(4))
if validation_table['sharpe'].notna().sum()==0:
    raise ValueError('No valid validation Sharpe values; inspect signals and data before continuing.')
BEST=validation_table.iloc[0]
BEST_LOOKBACK=int(BEST['lookback']); BEST_ENTRY=float(BEST['entry'])
print('Locked test configuration:',{'lookback':BEST_LOOKBACK,'entry':BEST_ENTRY,'exit':0.5,'stop_z':4,'max_hold':20,'cost_bp':5})

## 6. Untouched out-of-sample test (2024–2025)
The chosen pair and parameter configuration are now frozen. Historical data before 2024 is used only for causal indicator warm-up; the performance evaluation starts in 2024.

In [ ]:
full=backtest(prices,Y,X,lookback=BEST_LOOKBACK,entry=BEST_ENTRY,
              exit=0.5,stop_z=4,max_hold=20,cost_bp=5)
test_run=full.loc['2024-01-01':].copy()
summary=pd.DataFrame({'gross':metrics(test_run,'gross_return'),
                      'net':metrics(test_run,'net_return')})
summary.to_csv(OUT/'out_of_sample_metrics.csv')
test_run.to_csv(OUT/'out_of_sample_daily_backtest.csv',index_label='Date')
display(summary.round(5))
print('Caution: one historical test interval is not proof of durable alpha.')

## 7. Figures: price, spread / z-score, cumulative P&L and drawdown

In [ ]:
fig,ax=plt.subplots(figsize=(11,4))
(prices[[Y,X]]/prices[[Y,X]].iloc[0]).plot(ax=ax)
ax.set(title='Normalized adjusted prices (2018 = 1)',ylabel='Indexed price');fig.tight_layout()
fig.savefig(OUT/'01_normalized_prices.png',dpi=160);plt.show()

fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
test_run['spread'].plot(ax=axes[0],title=f'Spread: {Y} minus rolling hedge × {X}')
test_run['zscore'].plot(ax=axes[1],label='Z-score')
for h in [BEST_ENTRY,-BEST_ENTRY]:axes[1].axhline(h,ls='--',alpha=.65,color='tab:red')
axes[1].axhline(0,color='gray',lw=.7)
entries=(test_run['signal']!=0)&(test_run['signal'].shift(1).fillna(0)==0)
exits=(test_run['signal']==0)&(test_run['signal'].shift(1).fillna(0)!=0)
axes[1].scatter(test_run.index[entries],test_run.loc[entries,'zscore'],marker='^',s=40,label='Entry')
axes[1].scatter(test_run.index[exits],test_run.loc[exits,'zscore'],marker='x',s=40,label='Exit')
axes[1].legend();fig.tight_layout();fig.savefig(OUT/'02_spread_zscore_trades.png',dpi=160);plt.show()

nav=(1+test_run[['gross_return','net_return']]).cumprod()
fig,ax=plt.subplots(figsize=(11,4));nav.plot(ax=ax);ax.set(title='Out-of-sample cumulative wealth (initial = 1)',ylabel='Wealth index')
fig.tight_layout();fig.savefig(OUT/'03_cumulative_pnl.png',dpi=160);plt.show()

dd=nav['net_return']/nav['net_return'].cummax()-1
fig,ax=plt.subplots(figsize=(11,3));dd.plot(ax=ax,color='tab:red');ax.set(title='Out-of-sample net drawdown',ylabel='Drawdown')
fig.tight_layout();fig.savefig(OUT/'04_drawdown.png',dpi=160);plt.show()

## 8. Transaction-cost and sensitivity analysis
The grid below is **diagnostic on the test period**, not a second optimization stage. Do **not** select a new preferred strategy based on this table and then report it as untouched out-of-sample performance. It explores entry, exit, lookback, cost and stop thresholds.

In [ ]:
sensitivity=[]
scenarios=[]
for entry in [1.5,2.,2.5]: scenarios.append(('entry',entry,dict(entry=entry)))
for ex in [0.,0.5,1.]: scenarios.append(('exit',ex,dict(exit=ex)))
for lb in [60,120,252]: scenarios.append(('lookback',lb,dict(lookback=lb)))
for bp in [0,5,10]: scenarios.append(('cost_bp',bp,dict(cost_bp=bp)))
for stop in [3.,4.,5.]: scenarios.append(('stop_z',stop,dict(stop_z=stop)))
for parameter,value,override in scenarios:
    cfg=dict(lookback=BEST_LOOKBACK,entry=BEST_ENTRY,exit=.5,stop_z=4.,max_hold=20,cost_bp=5.)
    cfg.update(override)
    s=backtest(prices,Y,X,**cfg).loc['2024-01-01':]
    sensitivity.append({'parameter':parameter,'value':value,**metrics(s)})
sensitivity=pd.DataFrame(sensitivity)
sensitivity.to_csv(OUT/'test_sensitivity_diagnostics.csv',index=False)
display(sensitivity[['parameter','value','annual_return','sharpe','max_drawdown','trade_entries']].round(4))

## 9. Trading behaviour and year-by-year stability

In [ ]:
s=test_run['signal']
starts=(s!=0)&(s.shift(1).fillna(0)==0)
ends=(s==0)&(s.shift(1).fillna(0)!=0)
# completed trade intervals (position formed at entry date, first return next day)
entry_dates=list(test_run.index[starts]); exit_dates=list(test_run.index[ends])
trades=[]
for ed in entry_dates:
    candidates=[d for d in exit_dates if d>ed]
    if not candidates: continue
    xd=candidates[0]
    seg=test_run.loc[(test_run.index>ed)&(test_run.index<=xd),'net_return']
    trades.append({'entry':ed,'exit':xd,'holding_days':len(seg),
                   'net_trade_return':(1+seg).prod()-1})
trades=pd.DataFrame(trades)
trades.to_csv(OUT/'completed_trades.csv',index=False)
if len(trades):
    print('Average holding days:',round(trades.holding_days.mean(),2),
          '| Hit ratio:',round((trades.net_trade_return>0).mean(),3),
          '| Mean net trade return:',round(trades.net_trade_return.mean(),5))
else: print('No completed trades; trade-level metrics not available.')
annual=[]
for year,group in test_run.groupby(test_run.index.year):
    annual.append({'year':year,**metrics(group)})
annual=pd.DataFrame(annual)
annual.to_csv(OUT/'annual_test_metrics.csv',index=False)
display(annual.round(4))

## 10. Diagnostics, limitations and report prompts
**Questions to answer using the actual outputs:**
1. Which pair was selected, and what are its training-period cointegration statistics?
2. Does the training spread appear mean reverting? Is this property stable in validation and test periods?
3. Does the test strategy survive **5–10 bp** transaction costs?
4. Is the out-of-sample net Sharpe supported by enough trades, or is it statistically fragile?
5. How sensitive is performance to entry/exit thresholds, lookback windows and stop rules?
6. What could invalidate the result: multiple testing, structural breaks, sector shocks, borrowing constraints, data adjustments, slippage, daily-close execution, leverage and financing?

**Important distinctions:**
- Training cointegration p-values are **not** an out-of-sample trading-performance guarantee.
- A backtest on adjusted daily closes is **not** evidence of executable live returns.
- Pair selection across 10 candidates entails selection bias; consider multiple-testing adjustments or a later independent sample.
- This prototype applies target weights at each close for the following close-to-close interval; a production simulation should explicitly model order placement, fills, and funding.
- The stop-loss implemented here is a **Z-score threshold**, not a portfolio drawdown stop. A separate drawdown-based risk limit can be added.
- Position normalization assumes gross exposure of 1 when active, but beta and dollar neutrality are not identical. Rebalancing turnover is explicitly costed.

**Next research improvements:** walk-forward pair reselection, bootstrap confidence intervals, stability of cointegration, more realistic execution/borrow model, benchmark against passive exposure, and independent replication with another price vendor.

## Export manifest
When all cells complete, the folder `pairs_trading_outputs/` contains the cleaned price data, training pair ranking, validation selection, test metrics, trade log, sensitivity diagnostics, and four figures. These are the source materials for the separate short research report.

**Data source:** Yahoo Finance through `yfinance` (third-party data; review vendor terms and corporate-action handling).  
**Packages:** pandas, NumPy, SciPy, statsmodels, Matplotlib, yfinance.